In [ ]:
!python --version

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT:", ROOT)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *

### Defaults

In [ ]:
ROOT_DATA = ROOT / "data"

gdc = GDC(ROOT_DATA0=ROOT_DATA, ROOT_SRC=ROOT_SRC)

os.listdir(ROOT_DATA)[:10]

In [ ]:
os.listdir(gdc.root_data)[:10]

### Get all programs

In [ ]:
force = False
verbose = True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)

### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
prog_id = "TCGA"

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.head(3)

### GTEx

In [ ]:
df_gtex = gdc.read_GTEx_to_TCGA_table()
print(len(df_gtex))

df_gtex.head(3)

In [ ]:
verbose = True

psi_id = "TCGA-BRCA"
psi_id = "TCGA-LUAD"

gtex_id, gtex_tissue_ids = gdc.find_GTEx_to_TCGA_row(psi_id=psi_id, verbose=verbose)
print(">>>", gtex_id, gtex_tissue_ids)

### Get df_tumor - entropy - to find homogeneous rows

In [30]:
verbose = False

df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(psi_id=psi_id, verbose=verbose)

len(df_tumor), len(df_normal)

......40.........50.........60.........70.........80.........90.........100........


(240796, 240796)

In [ ]:
df_tumor.head(3)

In [ ]:
df_tumor2 = gdc.add_entropy(df_tumor)
print(len(df_tumor2))
df_tumor2.head(3)

In [ ]:
tumor_cols = [c for c in df_tumor.columns if c.startswith("tumor_")]

df_tumor3 = gdc.select_top_entropy(df_tumor2, q=0.1)
print(len(df_tumor3))
df_tumor3.head(3)

In [ ]:
geneid_list = list(np.unique(df_tumor3.gene_id))
len(geneid_list)

### GTEx portal - data manually downloaded

In [ ]:
print(len(geneid_list))
np.array(geneid_list[:50])

In [ ]:
gdc.gtex_id

In [ ]:
verbose = False
force = False

want_run = False

if want_run:
    df_gtex = gdc.get_gtex_TPM_expression_for_geneid_list(
        geneid_list=geneid_list,
        datasetId="gtex_v8",
        page_size=1000,
        sleep=0.1,
        timeout=60,
        force=force,
        verbose=verbose,
    )
else:
    df_gtex = pd.DataFrame()

print(df_gtex.shape)

### GTEx download

https://www.gtexportal.org/home/downloads/adult-gtex/overview

In [ ]:
url_counts = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
)

url_tpm = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct.gz"
)

url_meta = (
    "https://storage.googleapis.com/adult-gtex/metadata/"
    "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
)


filename_counts = gdc.root_gtex / gdc.fname_GTEx_counts
filename_meta = gdc.root_gtex / gdc.fname_GTEx_meta
filename_pheno = gdc.root_gtex / gdc.fname_GTEx_pheno

# --> download manually at https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
#                      and https://www.gtexportal.org/home/downloads/adult-gtex/metadata

# gdc.download_file(url_counts, filename_counts)
# gdc.download_file(url_meta, filename_meta)

### Phenotype - DTHHRDY (Hardy Scale of Death)

> It indicates how the donor died and how fast, which affects tissue quality.  


| Value	| Meaning  |
|---------|-----------|
| 0	| Ventilator case (on life support before death) |
| 1	| Violent and fast death (<10 min) |
| 2	| Fast death (10 min – 1 hour) |
| 3	| Intermediate death (1 – 24 hours) |
| 4	| Slow death (>24 hours, prolonged illness) |
| NA	| Unknown |


In [ ]:
df_pheno = gdc.read_GTEx_pheno(verbose=verbose)

print(len(df_pheno))
df_pheno.head(3)

In [ ]:
df_meta = gdc.read_GTEx_metadata(verbose=verbose)

print(len(df_meta))
df_meta.head(3)

### Healthy donors

1. Filter the tissue
2. Filter for high quality (Hardy Scale 1 or 2 are 'fast' deaths, less stress)
  . DTHHRDY: 1 = Ventilator, 2 = Fast death of natural causes
3. Sort by RIN score (SMRIN) to get the best preserved RNA

In [ ]:
df_meta_ctrl = gdc.prepare_gtex_df_control()

print(df_meta_ctrl.shape)
df_meta_ctrl.head(5)

In [ ]:
verbose = False

psi_id = "TCGA-LUSC"
psi_id = "TCGA-BRCA"
psi_id = "TCGA-LUAD"

df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(psi_id=psi_id, verbose=verbose)
len(df_tumor), len(df_normal)

In [ ]:
df_tumor.head(3)

In [ ]:
df_normal.head(3)

In [ ]:
force = False
verbose = False

df_gtex_ctrl, df_meta_ctrl = gdc.calc_gtex_control(
    psi_id=psi_id, Nsamples=15, force=force, verbose=verbose
)
print(df_gtex_ctrl.shape)

df_gtex_ctrl.head(3)

In [ ]:
verbose = False

run_loop = False

if run_loop:
    for i, row in df_psi.iterrows():
        psi_id = row.psi_id
        primary_site = row.primary_site
        print(f">>> {psi_id} - {primary_site[:15]}", end=" ")

        df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(
            psi_id=psi_id, verbose=verbose
        )

        if df_tumor.empty:
            print("No tumor expression data found")
            continue

        df_gtex_ctrl, df_meta_ctrl = gdc.calc_gtex_control(
            psi_id=psi_id, Nsamples=15, force=force, verbose=verbose
        )

        print(
            f"-> Tumor: {df_tumor.shape[0]}, Normal: {df_normal.shape[0]}, GTEx: {df_gtex_ctrl.shape[0]}"
        )

In [ ]:
gdc.root_data, ROOT_DATA, ROOT_SRC

In [ ]:
force = True
verbose = False

psi_id = "TCGA-LUSC"
psi_id = "TCGA-LUAD"
psi_id = "TCGA-BRCA"

df_degs, df_lfc, degs_txt, msg = gdc.calc_degs(
    psi_id=psi_id,
    root_scr=ROOT_SRC,
    run_conda=True,
    lfc_cutoff=1.0,
    fdr_cutoff=0.05,
    method="deseq2",
    verbose=verbose,
    force=force,
)

print(msg)
len(df_degs), len(df_lfc)

In [ ]:
df_lfc.head(3)

In [ ]:
df_degs.head(3)

In [ ]:
df_degs.tail(6)

In [ ]:
gdc.df_tumor

In [ ]:
gdc.df_normal

### Compare Normal Tissue x GTEx Controls

In [ ]:
def calc_degs(
    self,
    psi_id: str,
    root_scr: Path,
    run_conda: bool = False,
    lfc_cutoff: float = 1.0,
    fdr_cutoff: float = 0.05,
    method: str = "edger",
    force: bool = False,
    verbose: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, str, str]:

    self.set_primary_site(psi_id=psi_id)

    df_tumor, df_normal = self.get_file_expression_both_tumor_and_normal(
        psi_id=psi_id, verbose=verbose
    )
    if df_tumor.empty:
        if verbose:
            print(f"No tumor expression data found for {psi_id}")
        return pd.DataFrame(), pd.DataFrame(), "", ""

    df_gtex_ctrl, _ = self.calc_gtex_control(
        psi_id=psi_id, Nsamples=15, force=False, verbose=verbose
    )

    fname_degs = self.fname_degs % self.psi_id
    fname_lfc = self.fname_lfc % self.psi_id
    fname_degs_txt = self.fname_degs_txt % self.psi_id
    fname_sample_txt = self.fname_sample_txt % self.psi_id

    filename_degs = self.root_psi / fname_degs
    filename_lfc = self.root_psi / fname_lfc
    # filename_degs_txt = self.root_psi / fname_degs_txt
    # filename_sample_txt = self.root_psi / fname_sample_txt

    if filename_degs.exists() and filename_lfc.exists() and not force:
        try:
            df_degs = pdreadcsv(fname_degs, self.root_psi, verbose=verbose)
            df_lfc = pdreadcsv(fname_lfc, self.root_psi, verbose=verbose)
            degs_txt = read_txt(fname_degs_txt, self.root_psi, verbose=verbose)
            sample_txt = read_txt(fname_sample_txt, self.root_psi, verbose=verbose)

            return df_degs, df_lfc, degs_txt, sample_txt
        except:
            pass

    # geneid, symbol, type, samples
    min_N_cols = 3 + 2
    if df_normal.shape[1] < min_N_cols and df_gtex_ctrl.shape[1] < min_N_cols:
        msg = "Error: Normal samples and GTEx control do not have enough samples."
        return pd.DataFrame(), pd.DataFrame(), "", msg

    cdegs = CALC_DEGS(root_psi=self.root_psi, root_scr=root_scr, run_conda=run_conda)

    df_normal = cdegs.deduplicate_by_max_reads(df_normal)

    if df_normal.shape[1] < min_N_cols:
        msg = f"not enough normal samples --> substituting with GTEx control {df_normal.shape[1] - 3}."
        df_normal = self.prepare_gtex(df_gtex_ctrl)
        df_normal = cdegs.deduplicate_by_max_reads(df_normal)
    else:
        msg = f"enough normal samples {df_normal.shape[1] - 3}."

    msg = f"There are {df_tumor.shape[1] - 3} tumor samples; {msg}"

    df_tumor = cdegs.deduplicate_by_max_reads(df_tumor)
    self.df_tumor = df_tumor

    if method == "deseq2":
        if df_tumor.shape[0] < 2:
            print("Error: Tumor expression data has fewer than 2 samples.")
            return pd.DataFrame(), pd.DataFrame(), "", ""

        if df_normal.shape[0] < 2:
            print("Error: Normal expression data has fewer than 2 samples.")
            return pd.DataFrame(), pd.DataFrame(), "", ""

    df_lfc = cdegs.run_deg_rscript(
        df_tumor=df_tumor,
        df_normal=df_normal,
        method=method,
        manual_dispersion=0.1,
        min_total_count=10,
        merge_how="inner",
        keep_temp=False,
    )

    df_lfc = df_lfc.rename(columns={"log2FoldChange": "lfc", "padj": "fdr"})

    df_degs = df_lfc[(df_lfc.lfc >= lfc_cutoff) & (df_lfc.fdr < fdr_cutoff)].copy()
    df_degs.reset_index(drop=True, inplace=True)

    _ = pdwritecsv(df_lfc, fname_lfc, self.root_psi)
    _ = pdwritecsv(df_degs, fname_degs, self.root_psi)

    degs_txt = "\n".join(df_degs.symbol)
    _ = write_txt(degs_txt, fname_degs_txt, self.root_psi)

    _ = write_txt(msg, fname_sample_txt, self.root_psi)

    return df_degs, df_lfc, degs_txt, msg